# Phase 2: Text Preprocessing & Class Imbalance Mitigation

**Objective:**
Transform the raw, noisy text into a clean, standardized format suitable for both Deep Learning (LSTM/Transformers) and Traditional Machine Learning models. This pipeline ensures data integrity by handling missing values, duplicates, and linguistic anomalies before applying balancing techniques.

**Workflow:**
1. Initial Data Health Check (Drop raw NaNs & exact duplicates)
2. Regex-based Text Purification (URLs, mentions, special characters)
3. Post-cleaning Quality Assurance (Remove empty strings & single-word outliers)
4. Data Splitting & Class Weight Calculation

## 1. Initial Data Health Inspection
Before executing any drop operations, it is critical to inspect the exact volume and nature of missing values (NaNs) and duplicated rows to prevent unintended data loss and detect potential scraping bugs.

In [18]:
# ==========================================
# 0. ENVIRONMENT SETUP & IMPORTS
# ==========================================
import pandas as pd
import numpy as np 
import re
import warnings  

# Suppress minor warnings for clean output
warnings.filterwarnings('ignore')

# Define standard file paths
RAW_DATA_PATH = "../data/raw/text.csv"
PROCESSED_DATA_PATH = "../data/processed/"

print("Environment ready. Libraries imported.")  

Environment ready. Libraries imported.


In [19]:
# ==========================================
# 1. INITIAL DATA HEALTH INSPECTION
# ==========================================

print("Loading raw dataset into memory...") 
df_raw = pd.read_csv(RAW_DATA_PATH)
initial_shape = df_raw.shape

print("=== RAW DATA HEALTH REPORT ===")
print(f"Initial total rows: {df_raw.shape[0]}\n")

# 1.1 CHECK FOR MISSING VALUES (NaNs)
missing_counts = df_raw.isnull().sum()
print("--- 1. Missing Values ---")
print(missing_counts[missing_counts > 0])

# Inspect a sample of missing rows if any exist
if missing_counts.sum() > 0:
    print("\n[Sample of rows with Missing Values]:")
    print(df_raw[df_raw.isnull().any(axis=1)].head(5).to_string())
else:
    print("\n-> Excellent: No missing values (NaN) detected!")

# 1.2 CHECK FOR EXACT DUPLICATES
duplicate_count = df_raw.duplicated(subset=['text']).sum() 
print(f"\n--- 2. Duplicated Rows ---") 
print(f"Detected {duplicate_count} duplicated records.")

# Inspect a sample of duplicated pairs to verify
if duplicate_count > 0:
    print("\n[Sample of Duplicated Rows (Side-by-Side)]:")
    # keep=False to display both the original and the duplicated rows
    duplicates_sample = df_raw[df_raw.duplicated(subset=['text'], keep=False)].sort_values(by='text').head(6)
    print(duplicates_sample.to_string())
else:
    print("\n-> Excellent: No duplicated data detected!")

Loading raw dataset into memory...
=== RAW DATA HEALTH REPORT ===
Initial total rows: 416809

--- 1. Missing Values ---
Series([], dtype: int64)

-> Excellent: No missing values (NaN) detected!

--- 2. Duplicated Rows ---
Detected 22987 duplicated records.

[Sample of Duplicated Rows (Side-by-Side)]:
        Unnamed: 0          text  label
180753      180753    as a child      3
21541        21541    as a child      4
390789      390789     at school      4
270857      270857     at school      3
241108      241108  doesnt apply      4
138410      138410  doesnt apply      3


In [20]:
# ==========================================
# 1.3 EXECUTE CRUDE CLEANING (DROP NANS & DUPLICATES)
# ==========================================
print("\nExecuting crude cleaning...")

# 1. Drop complete NaNs to prevent regex errors later
df_clean = df_raw.dropna(subset=['text', 'label']).copy()
missing_dropped = df_raw.shape[0] - df_clean.shape[0]
 
# 2. Drop exact text duplicates to resolve conflicting labels
temp_shape = df_clean.shape[0]
df_clean = df_clean.drop_duplicates(subset=['text'], keep='first')
duplicates_dropped = temp_shape - df_clean.shape[0]

print(f"Successfully dropped {missing_dropped} missing (NaN) rows.")
print(f"Successfully dropped {duplicates_dropped} exact duplicate rows.")
print(f"Dataset shape ready for Regex pipeline: {df_clean.shape}")


Executing crude cleaning...
Successfully dropped 0 missing (NaN) rows.
Successfully dropped 22987 exact duplicate rows.
Dataset shape ready for Regex pipeline: (393822, 3)


## 2. Deep Learning Surface Cleaning (Digital Noise & Normalization)
Before altering the text, we must inspect the dataset for digital artifacts (URLs, HTML tags, @mentions). For Deep Learning models (BERT/LSTM), we aim to remove these meaningless artifacts while preserving semantic structures (lowercase, expanded contractions, and expressive punctuations like `! ? . ,`).

### 2.1 Inspect Digital Noise Footprints

In [21]:
# ==========================================
# 2.1 INSPECT DIGITAL NOISE (PRE-REGEX)
# ==========================================
import re

print("Scanning dataset for digital noise footprints...\n") 

# Define basic patterns to detect common digital noise
url_pattern = r'http[s]?://\S+|www\.\S+'
mention_pattern = r'@\w+'
html_pattern = r'&\w+;|<[^>]+>'

# Generate boolean masks to find rows containing these patterns
url_mask = df_clean['text'].str.contains(url_pattern, regex=True, na=False)
mention_mask = df_clean['text'].str.contains(mention_pattern, regex=True, na=False)
html_mask = df_clean['text'].str.contains(html_pattern, regex=True, na=False)

print("=== NOISE FOOTPRINT REPORT ===")
print(f"Contains URLs     : {url_mask.sum()} rows")
print(f"Contains Mentions : {mention_mask.sum()} rows")
print(f"Contains HTML tags: {html_mask.sum()} rows")

# Inspect actual samples if noise is detected
if url_mask.sum() > 0:
    print("\n[Sample] Texts with URLs:")
    print(df_clean[url_mask]['text'].head(3).to_string())

if mention_mask.sum() > 0:
    print("\n[Sample] Texts with @Mentions:")
    print(df_clean[mention_mask]['text'].head(3).to_string())
    
if html_mask.sum() > 0:
    print("\n[Sample] Texts with HTML artifacts:")
    print(df_clean[html_mask]['text'].head(3).to_string()) 

Scanning dataset for digital noise footprints...

=== NOISE FOOTPRINT REPORT ===
Contains URLs     : 0 rows
Contains Mentions : 0 rows
Contains HTML tags: 0 rows


### 2.2 Execute Deep Learning Text Purification
Based on the inspection, we apply a targeted regex engine. Unlike traditional ML preprocessing, we DO NOT remove stopwords or apply stemming. We preserve the natural grammar and structural punctuations necessary for Transformer self-attention mechanisms.

In [22]:
# ==========================================
# 2.2 EXECUTE DL TEXT PURIFICATION
# ==========================================
from tqdm import tqdm

# Enable pandas progress bar functionality
tqdm.pandas(desc="Purifying DL Text Data")

# Dictionary to expand common English contractions to preserve context (especially 'not')
CONTRACTION_MAP = {
    "i'm": "i am", "can't": "cannot", "won't": "will not",
    "don't": "do not", "doesn't": "does not", "didn't": "did not",
    "isn't": "is not", "aren't": "are not", "wasn't": "was not",
    "weren't": "were not", "haven't": "have not", "hasn't": "has not",
    "hadn't": "had not", "it's": "it is", "that's": "that is"
}

def clean_text_dl(text: str) -> str:
    """
    Purifies text explicitly for Deep Learning models.
    Removes digital noise, expands contractions, lowers cases, 
    but keeps alphabets and essential punctuations (! ? . ,)
    """
    text = str(text)
    
    # 1. Convert to lowercase (Syncs with bert-base-uncased architectures)
    text = text.lower()
        
    # 2. Expand contractions using the predefined dictionary
    for contraction, expansion in CONTRACTION_MAP.items():
        text = re.sub(r"\b" + contraction + r"\b", expansion, text)
            
    # 3. Remove URLs, mentions, and HTML tags
    text = re.sub(r'http[s]?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'&\w+;|<[^>]+>', ' ', text)
    
    # 4. Keep ONLY alphabets, spaces, and basic expressive punctuations (! ? . ,)
    text = re.sub(r'[^a-z\s\!\?\.\,]', ' ', text)
    
    # 5. Collapse multiple consecutive spaces into a single space and strip edges
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print(f"[INFO] Applying DL-Optimized Regex engine to {df_clean.shape[0]} rows...")

# Create a new column 'clean_text' to preserve the original for comparison
df_clean['clean_text'] = df_clean['text'].progress_apply(clean_text_dl)

print("\nDL Text purification completed. Sample output:") 
print(df_clean[['text', 'clean_text']].head(5).to_string())  

[INFO] Applying DL-Optimized Regex engine to 393822 rows...


Purifying DL Text Data: 100%|██████████| 393822/393822 [00:15<00:00, 26056.42it/s]


DL Text purification completed. Sample output:
                                                                                                                                                                                                                                         text                                                                                                                                                                                                                                  clean_text
0                                                                                                                                                                                               i just feel really helpless and heavy hearted                                                                                                                                                                                               i just feel really helpless and heavy hearted
1  i

## 3. Post-Cleaning Quality Assurance (QA)
The regex engine may have reduced some texts into empty strings (ghost texts) or left us with ultra-short sentences. For Deep Learning, sentences with fewer than 3 words often lack sufficient semantic context for the self-attention mechanism and can introduce label noise. We will inspect and filter these out.

### 3.1 Inspect Post-Cleaning Anomalies

In [23]:
# ==========================================
# 3.1 INSPECT POST-CLEANING ANOMALIES
# ==========================================
print("Scanning for post-cleaning anomalies...\n") 

# Calculate word count based on the newly cleaned text
df_clean['word_count_clean'] = df_clean['clean_text'].apply(lambda x: len(str(x).split()))

# Detect empty strings (ghost texts: word count is 0)
empty_mask = df_clean['word_count_clean'] == 0

# Detect ultra-short sentences (1 or 2 words)
short_mask = (df_clean['word_count_clean'] > 0) & (df_clean['word_count_clean'] < 3)

print("=== POST-CLEANING ANOMALY REPORT ===")
print(f"Empty strings (Ghost texts) : {empty_mask.sum()} rows")
print(f"Ultra-short sentences (< 3) : {short_mask.sum()} rows")

# Inspect samples of empty strings
if empty_mask.sum() > 0:
    print("\n[Sample] Ghost texts (became empty after regex):")
    print(df_clean[empty_mask][['text', 'clean_text', 'label']].head(3).to_string())

# Inspect samples of ultra-short sentences
if short_mask.sum() > 0:
    print("\n[Sample] Ultra-short sentences:")
    print(df_clean[short_mask][['text', 'clean_text', 'label']].head(5).to_string())
else:
    print("\n-> No ultra-short sentences found!") 

Scanning for post-cleaning anomalies...

=== POST-CLEANING ANOMALY REPORT ===
Empty strings (Ghost texts) : 0 rows
Ultra-short sentences (< 3) : 84 rows

[Sample] Ultra-short sentences:
                     text          clean_text  label
28                   when                when      3
6689                   in                  in      0
11417  before examination  before examination      4
18037           at hostel           at hostel      3
21441         no response         no response      3


### 3.2 Execute Quality Assurance Filtering

In [24]:
# ==========================================
# 3.2 EXECUTE POST-CLEANING QA (FILTERING)
# ==========================================
print("Executing Post-cleaning QA...")

initial_qa_shape = df_clean.shape[0]

# Filter out empty strings and short sentences (Keep only rows with >= 3 words)
df_clean = df_clean[df_clean['word_count_clean'] >= 3]

dropped_total = initial_qa_shape - df_clean.shape[0]

# Drop the temporary word count column to keep the dataframe clean
df_clean = df_clean.drop(columns=['word_count_clean'])

print(f"Successfully purged {dropped_total} anomalous rows (empty or < 3 words).")
print(f"Final purified dataset shape ready for splitting: {df_clean.shape}") 

Executing Post-cleaning QA...
Successfully purged 84 anomalous rows (empty or < 3 words).
Final purified dataset shape ready for splitting: (393738, 4)


## 4. Data Splitting & Class Imbalance Mitigation
Before partitioning the dataset, we must inspect the target label distribution to understand the severity of class imbalance. Following this, we will perform a stratified split (80% Train, 10% Validation, 10% Test) to preserve this exact distribution across all sets. Finally, we generate PyTorch class weights and an undersampled baseline set to assist the downstream Deep Learning models.

### 4.1 Inspect Label Distribution

In [25]:
# ==========================================
# 4.1 INSPECT CLASS DISTRIBUTION
# ==========================================
print("Inspecting Target Label Distribution...\n") 

# Calculate absolute counts and percentages
label_counts = df_clean['label'].value_counts().sort_index()
label_percentages = df_clean['label'].value_counts(normalize=True).sort_index() * 100

print("=== CLASS DISTRIBUTION REPORT ===")
for label in label_counts.index:
    print(f"Label {label}: {label_counts[label]:>6} rows ({label_percentages[label]:>5.2f}%)")

print("\n-> Note: High imbalance detected. Stratified splitting and Class Weights are mandatory.") 

Inspecting Target Label Distribution...

=== CLASS DISTRIBUTION REPORT ===
Label 0: 118491 rows (30.09%)
Label 1: 135018 rows (34.29%)
Label 2:  29468 rows ( 7.48%)
Label 3:  54744 rows (13.90%)
Label 4:  43610 rows (11.08%)
Label 5:  12407 rows ( 3.15%)

-> Note: High imbalance detected. Stratified splitting and Class Weights are mandatory.


### 4.2 Execute Stratified Splitting & Export Artifacts

In [26]:
# ==========================================
# 4.2 EXECUTE STRATIFIED SPLIT & EXPORT
# ==========================================
import os
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from imblearn.under_sampling import RandomUnderSampler

print("Initiating Stratified Data Splitting (80/10/10)...")

# Keep only the essential columns for Deep Learning input
X = df_clean[['clean_text']]
y = df_clean['label']

# Split 1: 80% Train, 20% Temp (for Val and Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
# Split 2: Divide Temp equally into Val (10%) and Test (10%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"Train Size: {X_train.shape[0]} rows")
print(f"Val Size  : {X_val.shape[0]} rows")
print(f"Test Size : {X_test.shape[0]} rows")

print("\nComputing Class Weights (PyTorch Tensor) from the Train set...")
classes = np.unique(y_train)
# 'balanced' mode automatically calculates weights inversely proportional to class frequencies
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
weights_tensor = torch.tensor(weights, dtype=torch.float)

print("Generating alternative Undersampled Train Set (for fast baseline testing)...")
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

# --- EXPORT ARTIFACTS ---
print("\nSaving all artifacts to disk...")
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

# Reconstruct DataFrames and rename 'clean_text' back to 'text' for standard model input
train_df = pd.concat([X_train, y_train], axis=1).rename(columns={'clean_text': 'text'})
val_df = pd.concat([X_val, y_val], axis=1).rename(columns={'clean_text': 'text'})
test_df = pd.concat([X_test, y_test], axis=1).rename(columns={'clean_text': 'text'})
train_under_df = pd.concat([X_train_under, y_train_under], axis=1).rename(columns={'clean_text': 'text'})

# Save to CSV
train_df.to_csv(f"{PROCESSED_DATA_PATH}train.csv", index=False)
val_df.to_csv(f"{PROCESSED_DATA_PATH}val.csv", index=False)
test_df.to_csv(f"{PROCESSED_DATA_PATH}test.csv", index=False)
train_under_df.to_csv(f"{PROCESSED_DATA_PATH}train_undersampled.csv", index=False)

# Save PyTorch Tensor
torch.save(weights_tensor, f"{PROCESSED_DATA_PATH}class_weights.pt")

print(f"Pipeline complete! All artifacts exported successfully to: {PROCESSED_DATA_PATH}")
print("=== TASK 2 PREPROCESSING COMPLETED ===")  

Initiating Stratified Data Splitting (80/10/10)...
Train Size: 314990 rows
Val Size  : 39374 rows
Test Size : 39374 rows

Computing Class Weights (PyTorch Tensor) from the Train set...
Generating alternative Undersampled Train Set (for fast baseline testing)...

Saving all artifacts to disk...
Pipeline complete! All artifacts exported successfully to: ../data/processed/
=== TASK 2 PREPROCESSING COMPLETED ===
